# models

> Configuration dataclass for viewport-fit measurement instances

In [ ]:
#| default_exp models

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass

## ViewportFitConfig

Central configuration for a viewport-fit measurement instance. Each instance manages a single target element, fitting its height to the remaining viewport space. Multiple instances can coexist on the same page via namespace isolation.

In [ ]:
#| export
@dataclass
class ViewportFitConfig:
    """Configuration for a viewport-fit measurement instance."""

    namespace: str  # Unique prefix for JS isolation (e.g., "source_select")
    target_id: str  # HTML ID of the element to fit to viewport height
    inner_id: str = ""  # Optional child element that also receives the calculated height
    container_id: str = ""  # Optional container element — measure against this instead of viewport
    min_height: int = 200  # Minimum height in pixels
    debounce_ms: int = 100  # Resize debounce delay in milliseconds
    scroll_to_top: bool = True  # Scroll to (0,0) before measuring (for HTMX SPAs)
    enable_htmx_settle: bool = True  # Remeasure on htmx:afterSettle events
    observe_siblings: bool = True  # Watch sibling elements for size changes via ResizeObserver
    resize_callback: str = ""  # Optional JS expression called after resize
    debug: bool = False  # Enable debug console logging by default

    @property
    def ns(self) -> str:  # Capitalized namespace for JS function names
        """Capitalized namespace for JS function names."""
        return "".join(p.capitalize() for p in self.namespace.split("_"))

    @property
    def handler_key(self) -> str:  # Window-level key for resize handler cleanup
        """Window-level key for resize handler cleanup."""
        return f"_vfResizeHandler_{self.namespace.replace('-', '_')}"

    @property
    def settle_handler_key(self) -> str:  # Window-level key for HTMX settle handler
        """Window-level key for HTMX settle handler cleanup."""
        return f"_vfSettleHandler_{self.namespace.replace('-', '_')}"

    @property
    def observer_key(self) -> str:  # Window-level key for ResizeObserver cleanup
        """Window-level key for sibling ResizeObserver cleanup."""
        return f"_vfSiblingObserver_{self.namespace.replace('-', '_')}"

    @property
    def debug_flag(self) -> str:  # JS window debug flag name
        """Window-level debug flag name."""
        return f"viewportFitDebug_{self.namespace.replace('-', '_')}"

    @property
    def log_prefix(self) -> str:  # Console log prefix string
        """Console log prefix string."""
        return f"vf:{self.namespace}"

    @property
    def recalc_fn(self) -> str:  # Global function name for manual recalculation
        """Global function name for manual recalculation."""
        return f"recalculate{self.ns}Height"

## Tests

In [ ]:
# Single-word namespace
cfg = ViewportFitConfig(namespace="demo", target_id="my-target")
assert cfg.ns == "Demo"
assert cfg.handler_key == "_vfResizeHandler_demo"
assert cfg.settle_handler_key == "_vfSettleHandler_demo"
assert cfg.observer_key == "_vfSiblingObserver_demo"
assert cfg.debug_flag == "viewportFitDebug_demo"
assert cfg.log_prefix == "vf:demo"
assert cfg.recalc_fn == "recalculateDemoHeight"

# Multi-word namespace
cfg2 = ViewportFitConfig(namespace="source_select", target_id="tss-grid")
assert cfg2.ns == "SourceSelect"
assert cfg2.recalc_fn == "recalculateSourceSelectHeight"
assert cfg2.handler_key == "_vfResizeHandler_source_select"

# Hyphenated namespace
cfg3 = ViewportFitConfig(namespace="card-stack", target_id="cs-0")
assert cfg3.handler_key == "_vfResizeHandler_card_stack"
assert cfg3.observer_key == "_vfSiblingObserver_card_stack"
assert cfg3.debug_flag == "viewportFitDebug_card_stack"

# Defaults
assert cfg.min_height == 200
assert cfg.debounce_ms == 100
assert cfg.scroll_to_top == True
assert cfg.enable_htmx_settle == True
assert cfg.observe_siblings == True
assert cfg.resize_callback == ""
assert cfg.inner_id == ""
assert cfg.container_id == ""
assert cfg.debug == False

# Container ID
cfg_c = ViewportFitConfig(namespace="cont", target_id="t", container_id="my-box")
assert cfg_c.container_id == "my-box"

print("All tests passed!")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()